# 13 · Ajuste fino de la CNN de placas con fotos reales (TensorFlow/Keras)

Se ejecuta en **Google Colab con GPU**. Parte de la CNN del notebook 12 (97,6 % por carácter en sintéticas, **0/10 en placas reales**) y la **ajusta con recortes reales**: la placa de la demo (`2602SIT`, fotos propias del equipo)
**y 6 de las 10 placas reales de otros textos**, mezclados con placas sintéticas nuevas, para cerrar la brecha sintético → real.

**Primera versión (solo la placa de la demo):** leyó la placa de la demo (7/7 en otro parachoques) pero **memorizó**: 3 de las 10 placas de otros textos salieron como `2602SIT` con confianza ≥ 0,93 (falsos positivos con el umbral de A4). Esta versión añade textos distintos para reducirlo.

**Qué se mide, y por qué importa el orden:**
1. **Sintéticas (300)**: que no se olvide lo aprendido.
2. **Recortes de prueba de la placa de la demo (7)**: fotos de **otro parachoques** que no se usan para entrenar (otro escenario, mismo texto). Mide si generaliza a una toma distinta, **no** si lee cualquier placa.
3. **4 placas reales de otros textos reservadas** (`2534RIS`, `5280YIN`, `3993YHA` y la antigua `510MDA`): nunca se usan para entrenar. Es la prueba honesta de que **lee** y no memoriza. Si sus lecturas salen `2602SIT`, memorizó. Las otras 6 sí se usan para entrenar.

**Archivos a subir:** `ml.zip`, `recortes_demo.zip` (28 recortes + `verdad.json`), `recortes.zip` (las 10 placas reales del notebook 10/11) y `placas_cnn.keras` (del `salida.zip` del notebook 12).
Los recortes y el modelo no se versionan (I6).

In [ ]:
import os, zipfile, json
from google.colab import files

subidos = files.upload()     # ml.zip + recortes_demo.zip + recortes.zip + placas_cnn.keras
for d in ("condohub-ia", "demo", "recortes"):
    os.makedirs(f"/content/{d}", exist_ok=True)
for nombre in subidos:
    n = nombre.lower()
    if n.endswith(".zip"):
        destino = "condohub-ia" if n.startswith("ml") else ("demo" if "demo" in n else "recortes")
        zipfile.ZipFile(nombre).extractall(f"/content/{destino}")
os.chdir("/content/condohub-ia")
demo = json.load(open("/content/demo/verdad.json", encoding="utf-8"))
verdad = json.load(open("/content/recortes/verdad.json", encoding="utf-8"))
IDX_PRUEBA = (1, 3, 6, 8)       # 2534RIS, 5280YIN, 510MDA (antigua), 3993YHA: reservadas, no se entrenan
verdad_p = [v for i, v in enumerate(verdad) if i in IDX_PRUEBA]
verdad_e = [v for i, v in enumerate(verdad) if i not in IDX_PRUEBA and len(v["real"]) == 7]
print(len(demo), "recortes de la placa de la demo |", len(verdad_e), "placas reales de otros textos para entrenar |", len(verdad_p), "reservadas para probar")

In [ ]:
import random, string, time, numpy as np, tensorflow as tf
from tensorflow import keras
from PIL import Image, ImageFilter, ImageEnhance
from ml.src import placas_sinteticas as ps, placas
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
SEMILLA = 7
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

ALTO, ANCHO = 32, 128
ALFA = string.digits + string.ascii_uppercase

def etiqueta(texto):
    return [ALFA.index(c) for c in texto]

def decodificar(prob):
    p = prob.copy(); p[:4, 10:] = 0; p[4:, :10] = 0
    p = p / p.sum(axis=1, keepdims=True)
    return "".join(ALFA[k] for k in p.argmax(1)), float(p.max(axis=1).prod())

def preparar(im, desde=0.4):
    im = im.convert("L")
    im = im.crop((0, int(desde * im.height), im.width, im.height))
    return np.asarray(im.resize((ANCHO, ALTO), Image.BICUBIC), dtype=np.float32) / 255.0

def generar_sinteticas(n, rng, fuerte=True):
    X = np.zeros((n, ALTO, ANCHO, 1), np.float32); Y = np.zeros((n, 7), np.int32)
    for i in range(n):
        t = ps.texto_aleatorio(rng)
        im = ps.dibujar(t, rng, degradar=True)
        if fuerte:
            w = rng.randint(90, 180)
            im = im.resize((w, w // 2), Image.BILINEAR).resize((360, 180), Image.BILINEAR)
            im = im.crop((rng.randint(0, 24), rng.randint(0, 12), 360 - rng.randint(0, 24), 180 - rng.randint(0, 12)))
        a = preparar(im, desde=rng.uniform(0.3, 0.5) if fuerte else 0.4)
        if fuerte:
            a = np.clip(a + np.random.normal(0, rng.uniform(0, 0.06), a.shape), 0, 1)
        X[i, :, :, 0] = a; Y[i] = etiqueta(t)
    return X, Y

## 1 · Datos reales: aumentar los recortes de la placa de la demo

Cada recorte real se convierte en muchas variantes: recorte irregular de bordes, giro, perspectiva, desenfoque, baja resolución, luz y ruido. Los recortes del grupo `prueba` **no se tocan**.

In [ ]:
def aumentar_real(im, rng):
    w, h = im.size
    m = lambda: rng.uniform(0, 0.08)
    im = im.crop((int(m() * w), int(m() * h), int(w * (1 - m())), int(h * (1 - m()))))
    w, h = im.size
    j = lambda: rng.uniform(-0.06, 0.06)
    cuad = (j() * w, j() * h,  j() * w, h * (1 + j()),  w * (1 + j()), h * (1 + j()),  w * (1 + j()), j() * h)   # perspectiva
    im = im.transform((w, h), Image.QUAD, cuad, Image.BICUBIC, fillcolor=(200, 200, 200))
    im = im.rotate(rng.uniform(-8, 8), resample=Image.BICUBIC, fillcolor=(200, 200, 200))
    if rng.random() < 0.6:
        k = rng.randint(80, 240)
        im = im.resize((k, max(8, int(k * h / w))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0, 1.6)))
    im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.55, 1.3))
    im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.6, 1.3))
    return im

def cargar(lista, carpeta):
    return [(Image.open(f"{carpeta}/{v['archivo']}").convert("RGB"), v["real"]) for v in lista]

ent = cargar([v for v in demo if v["grupo"] == "entrenamiento"], "/content/demo")
pru = cargar([v for v in demo if v["grupo"] == "prueba"], "/content/demo")
entrenar_otras = cargar(verdad_e, "/content/recortes")     # 6 placas de otros textos
ent = ent + entrenar_otras * 3                              # se repiten para pesar casi la mitad de las muestras reales
print(len(ent), "entradas de entrenamiento real |", len(pru), "de prueba de la demo")

def generar_reales(n, rng):
    X = np.zeros((n, ALTO, ANCHO, 1), np.float32); Y = np.zeros((n, 7), np.int32)
    for i in range(n):
        im, t = rng.choice(ent)
        a = preparar(aumentar_real(im, rng), desde=rng.uniform(0.3, 0.5))
        X[i, :, :, 0] = np.clip(a + np.random.normal(0, rng.uniform(0, 0.05), a.shape), 0, 1); Y[i] = etiqueta(t)
    return X, Y

rng = random.Random(SEMILLA)
N_SINT, N_REAL = 20000, 4000
Xs, Ys = generar_sinteticas(N_SINT, rng)
Xr, Yr = generar_reales(N_REAL, rng)
Xtr = np.concatenate([Xs, Xr]); Ytr = np.concatenate([Ys, Yr])
orden = np.random.permutation(len(Xtr)); Xtr, Ytr = Xtr[orden], Ytr[orden]
Xva, Yva = generar_sinteticas(2000, random.Random(SEMILLA + 1))
Xte, Yte = generar_sinteticas(300, random.Random(SEMILLA + 2), fuerte=False)
print(Xtr.shape, "| reales aumentados:", N_REAL, "de", len(Xtr))

## 2 · Medir ANTES del ajuste (el modelo del notebook 12)

In [ ]:
modelo = keras.models.load_model("placas_cnn.keras") if os.path.exists("placas_cnn.keras") else keras.models.load_model("/content/placas_cnn.keras")

def leer_lote(m, X):
    lect = [decodificar(p) for p in m.predict(X, verbose=0)]
    return [t for t, _ in lect], np.array([c for _, c in lect])

def exactitud_sint(m):
    t, _ = leer_lote(m, Xte)
    return placas.metricas_lectura(["".join(ALFA[k] for k in f) for f in Yte], t)

Xp = np.stack([preparar(im)[..., None] for im, _ in pru]); reales_p = [t for _, t in pru]
Xv = np.stack([preparar(Image.open(f"/content/recortes/{v['archivo']}"))[..., None] for v in verdad_p]); reales_v = [v["real"] for v in verdad_p]

def informe(m, etiqueta_):
    s = exactitud_sint(m)
    tp, cp = leer_lote(m, Xp)
    tv, cv = leer_lote(m, Xv)
    ex_p = sum(t == r for t, r in zip(tp, reales_p))
    ex_v = sum(t == r for t, r in zip(tv, reales_v))
    car_v = placas.metricas_lectura(reales_v, tv)["precision_por_caracter"]
    memorizo = sum(t == "2602SIT" for t in tv)
    print(f"[{etiqueta_}] sintéticas {s['placa_completa']:.1%} placa / {s['precision_por_caracter']:.1%} car | "
          f"demo (prueba) {ex_p}/{len(pru)} conf media {cp.mean():.2f} | "
          f"4 placas reservadas {ex_v}/{len(reales_v)}, {car_v:.1%} por carácter, lecturas '2602SIT': {memorizo}/{len(reales_v)}")
    return {"sinteticas": s, "demo_prueba": f"{ex_p}/{len(pru)}", "demo_conf_media": float(cp.mean()),
            "reservadas_exactas": int(ex_v), "reservadas_precision_caracter": car_v, "reservadas_lecturas_2602SIT": int(memorizo)}, tv, cv

antes, _, _ = informe(modelo, "antes")

## 3 · Ajuste fino

Tasa de aprendizaje baja (para no olvidar lo sintético). Se detiene con la validación sintética.

In [ ]:
modelo.compile(optimizer=keras.optimizers.Adam(3e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cb = [keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_loss"),
      keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)]
hist = modelo.fit(Xtr, Ytr, validation_data=(Xva, Yva), epochs=40, batch_size=128, callbacks=cb, verbose=2)
print("épocas:", len(hist.history["loss"]))

## 4 · Medir DESPUÉS

Lo importante: las fotos de prueba de la demo (otro parachoques) y, sobre todo, las **4 placas reservadas de otros textos**. Si «lecturas 2602SIT» es 0 y las exactas suben, aprendió a leer;
si «lecturas 2602SIT» es alto, solo memorizó la placa de la demo. (La antigua `510MDA` no cumple A4: no puede leerse bien.)

In [ ]:
despues, tv, cv = informe(modelo, "después")
for r, t, c in zip(reales_v, tv, cv):
    print(f"real {r:8s} leída {t}  conf {c:.2f}  {'OK' if t == r else 'x'}")
tp, cp = leer_lote(modelo, Xp)
print("\nfotos de prueba de la demo:", [f"{t} ({c:.2f})" for t, c in zip(tp, cp)])

## 5 · Exportar (TFLite fp32) y guardar métricas

In [ ]:
os.makedirs("/content/salida13", exist_ok=True)
modelo.save("/content/salida13/placas_cnn_ajustada.keras")
tfl = tf.lite.TFLiteConverter.from_keras_model(modelo).convert()
open("/content/salida13/placas_cnn_ajustada_fp32.tflite", "wb").write(tfl)

it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
ent_d, sal_d = it.get_input_details()[0], it.get_output_details()[0]
iguales = 0
for x, t_k in zip(Xp, tp):
    it.set_tensor(ent_d["index"], x[None]); it.invoke()
    iguales += decodificar(it.get_tensor(sal_d["index"])[0])[0] == t_k
print("TFLite fp32 igual a Keras en las fotos de prueba:", iguales, "/", len(Xp), "|", round(len(tfl) / 1e6, 2), "MB")

# Pesos para el navegador (ml/web/demo_webcam/cnn.mjs): BatchNorm plegada en cada convolucion; formato Keras (kh, kw, cin, cout).
partes, manifiesto, desplazamiento = [], [], 0
def agregar(nombre, a):
    global desplazamiento
    a = np.asarray(a, np.float32)
    manifiesto.append({"nombre": nombre, "forma": list(a.shape), "offset": desplazamiento}); partes.append(a.tobytes()); desplazamiento += a.nbytes
convs = [l for l in modelo.layers if isinstance(l, keras.layers.Conv2D)]
bns = [l for l in modelo.layers if isinstance(l, keras.layers.BatchNormalization)]
densas = [l for l in modelo.layers if isinstance(l, keras.layers.Dense)]
assert len(convs) == 3 and len(bns) == 3 and len(densas) == 2
for i, (c, b) in enumerate(zip(convs, bns)):
    W = c.get_weights()[0]; g, be, mu, var = b.get_weights()
    esc = g / np.sqrt(var + b.epsilon)
    agregar(f"conv{i}_w", W * esc); agregar(f"conv{i}_b", be - mu * esc)
for i, d in enumerate(densas):
    W, bb = d.get_weights(); agregar(f"dense{i}_w", W); agregar(f"dense{i}_b", bb)
open("/content/salida13/placas_cnn_pesos.bin", "wb").write(b"".join(partes))
json.dump(manifiesto, open("/content/salida13/placas_cnn_pesos.json", "w"))
muestra = Xp[:5]
json.dump({"entradas": [m.reshape(-1).tolist() for m in muestra], "salidas": [p.reshape(-1).tolist() for p in modelo.predict(muestra, verbose=0)]},
          open("/content/salida13/placas_cnn_paridad.json", "w"))
print("pesos para el navegador:", round(desplazamiento / 1e6, 2), "MB")

metricas = {"notebook": "13_ajuste_fino_placas_reales.ipynb", "base": "CNN del notebook 12", "tensorflow": tf.__version__, "semilla": SEMILLA,
            "datos": {"sinteticas_nuevas": N_SINT, "reales_aumentados": N_REAL, "entradas_reales_entrenamiento": len(ent), "recortes_demo_prueba": len(pru), "placas_otros_textos_entrenamiento": len(verdad_e), "placas_otros_textos_reservadas": len(verdad_p)},
            "antes": antes, "despues": despues, "tflite_fp32_mb": round(len(tfl) / 1e6, 2), "tflite_igual_keras_prueba": f"{iguales}/{len(Xp)}",
            "limites": "Pocas placas reales (la de la demo en dos escenarios y 6 de otros textos); 4 placas reservadas son la prueba de generalizacion."}
json.dump(metricas, open("/content/salida13/metrics.json", "w"), indent=2, ensure_ascii=False)
!cd /content && zip -qr salida13.zip salida13
from google.colab import files
files.download("/content/salida13.zip")
print(json.dumps(metricas, indent=2, ensure_ascii=False))